In [1]:
import pandas as pd
import numpy as np
import brightway2 as bw
import os
from pathlib import Path

from config import Config
from helpers import (
    process_all_impacts_results,
    process_results,
    process_timeline_results,
    process_combined_results,
    add_activity_metrics_vs_scenario,
    add_label,
    get_consuming_activities,
    close_results,
    get_baseline_sdf,
    process_non_grouped_results,
)

In [2]:
config = Config()
background_scenario = "remind - SSP1-PkBudg650"
sensitivity_analysis_scenario = "image - SSP1-VLLO"
impact_assessment_method = "EF v3.1"

bw.projects.set_current("Master thesis")
ei = bw.Database("ecoinvent-3.12-cutoff")

In [3]:
cow_milk_activities = [
    activity
    for activity in ei
    if "milk production" in activity["name"]
    and "cow" in activity["name"]
    and "cow milk" in activity["reference product"]
]
sdf_milk = get_baseline_sdf(cow_milk_activities)
milk_input_products = (
    sdf_milk[sdf_milk["flow type"] == "technosphere"]["from reference product"]
    .drop_duplicates()
    .to_list()
)

potato_activities = [
    activity
    for activity in ei
    if "potato production" in activity["name"]
    and activity["reference product"] == "potato"
]
sdf_potato = get_baseline_sdf(potato_activities)
potato_input_products = (
    sdf_potato[sdf_potato["flow type"] == "technosphere"]["from reference product"]
    .drop_duplicates()
    .to_list()
)

# Process timeline results

In [4]:
timeline_results = pd.DataFrame()

for scenario in [background_scenario, sensitivity_analysis_scenario]:
    path_results = f"../Results/{impact_assessment_method}/{scenario}/Timeline"
    list_files = files = [f for f in os.listdir(path_results) if f.endswith(".csv")]

    for raw_file in list_files:
        product = raw_file.split("_")[3]
        impact_category = raw_file.split("_")[-1].split(".")[0]
        raw_data = pd.read_csv(os.path.join(path_results, raw_file))
        product_timeline_results = process_timeline_results(
            raw_data, scenario, product, impact_category="climate change"
        )
        timeline_results = pd.concat([timeline_results, product_timeline_results])

timeline_results = timeline_results.sort_values(
    by=["scenario", "final_product_name", "year", "share"],
    ascending=[True, True, True, False],
).reset_index(drop=True)

timeline_results

,scenario_group,scenario,year,final_product_name,impact_category,reference_product_name,value,share
0,image - SSP1-VLLO,image - SSP1-VLLO,2025,cheese,climate change,cow milk,4.162678e+00,3.728649e-01
1,image - SSP1-VLLO,image - SSP1-VLLO,2025,cheese,climate change,"electricity, high voltage",1.690643e+00,1.514365e-01
2,image - SSP1-VLLO,image - SSP1-VLLO,2025,cheese,climate change,soybean,8.173694e-01,7.321450e-02
3,image - SSP1-VLLO,image - SSP1-VLLO,2025,cheese,climate change,"land tenure, arable land, measured as carbon n...",7.219066e-01,6.466357e-02
4,image - SSP1-VLLO,image - SSP1-VLLO,2025,cheese,climate change,alfalfa-grass silage,3.046915e-01,2.729223e-02
...,...,...,...,...,...,...,...,...
6472,remind - SSP1-PkBudg650,remind - SSP1-PkBudg650,2050,potato,climate change,"polyester resin, unsaturated",1.756847e-06,1.018363e-05
6473,remind - SSP1-PkBudg650,remind - SSP1-PkBudg650,2050,potato,climate change,"acetone, liquid",1.756172e-06,1.017972e-05
6474,remind - SSP1-PkBudg650,remind - SSP1-PkBudg650,2050,potato,climate change,purified terephthalic acid,1.732567e-06,1.004290e-05
6475,remind - SSP1-PkBudg650,remind - SSP1-PkBudg650,2050,potato,climate change,Rest (-),-1.298348e-07,-7.525924e-07


## Process a year 

In [5]:
years = ["2025", "2030", "2035", "2040", "2045", "2050"]
impact_results = pd.DataFrame()

BASE_DIR = Path.cwd().resolve()
RESULTS_DIR = BASE_DIR.parent / "Results"

for y in years:
    path_results = (
        RESULTS_DIR / impact_assessment_method / background_scenario / y
        if y != "2025"
        else RESULTS_DIR / impact_assessment_method / "Baseline"
    )

    list_files = [f for f in os.listdir(path_results) if f.endswith(".csv")]

    for raw_file in list_files:
        print("Processing file : ", raw_file)

        scenario_group = raw_file.split("_")[1]

        impact_category = raw_file.split("_")[-1].split(".")[0]
        year = raw_file.split("_")[2]
        product = raw_file.split("_")[3]

        is_combined_scenario = scenario_group.count("+") > 1

        raw_data = pd.read_csv(os.path.join(path_results, raw_file))

        if not is_combined_scenario:
            # Background results : all impact categories are processed

            if impact_category == "all":
                all_impacts_results_scenario = process_all_impacts_results(
                    raw_data, scenario_group, year, product
                )
                impact_results = pd.concat(
                    [impact_results, all_impacts_results_scenario]
                )

            # Foreground result : only carbon footprint is processed

            else:
                impact_results_scenario = process_results(
                    raw_data, scenario_group, year, product, impact_category
                )
                impact_results = pd.concat([impact_results, impact_results_scenario])

        else:
            impact_results_combined = process_combined_results(
                raw_data, scenario_group, year, product, impact_category
            )
            impact_results = pd.concat([impact_results, impact_results_combined])

full_year_results = impact_results.sort_values(
    by=["final_product_name", "scenario", "impact_category", "year", "share"],
    ascending=[True, True, True, True, False],
).reset_index(drop=True)

full_year_results

Processing file :  EF v3.1_baseline_2025_cheese_all.csv
Processing file :  EF v3.1_baseline_2025_potato_all.csv
Processing file :  EF v3.1_remind - SSP1-PkBudg650+agricultural-practices+electric-tractors_2030_potato_climate change.csv
Processing file :  EF v3.1_remind - SSP1-PkBudg650+agricultural-practices_2030_potato_climate change.csv
Processing file :  EF v3.1_remind - SSP1-PkBudg650+cf-agricultural-practices+electric-tractors+ruminant-methane-mitigation_2030_cheese_climate change.csv
Processing file :  EF v3.1_remind - SSP1-PkBudg650+cf-eef-fertilizers+electric-tractors+ruminant-methane-mitigation_2030_cheese_climate change.csv
Processing file :  EF v3.1_remind - SSP1-PkBudg650+cow-feed-agricultural-practices_2030_cheese_climate change.csv
Processing file :  EF v3.1_remind - SSP1-PkBudg650+cow-feed-eef-fertilizers_2030_cheese_climate change.csv
Processing file :  EF v3.1_remind - SSP1-PkBudg650+eef-fertilizers+electric-tractors_2030_potato_climate change.csv
Processing file :  EF 

,scenario_group,scenario,year,final_product_name,impact_category,reference_product_name,value,share
0,baseline,baseline,2025,cheese,acidification,cow milk,7.692643e-02,5.862745e-01
1,baseline,baseline,2025,cheese,acidification,"electricity, high voltage",8.858481e-03,6.751258e-02
2,baseline,baseline,2025,cheese,acidification,maize grain,7.326272e-03,5.583525e-02
3,baseline,baseline,2025,cheese,acidification,alfalfa-grass silage,5.293520e-03,4.034317e-02
4,baseline,baseline,2025,cheese,acidification,maize silage,3.742565e-03,2.852297e-02
...,...,...,...,...,...,...,...,...
63450,remind - SSP1-PkBudg650+eef-fertilizers+electr...,remind - SSP1-PkBudg650+urease_inhibitors+elec...,2050,potato,climate change,sewage sludge,1.541568e-06,1.044614e-05
63451,remind - SSP1-PkBudg650+eef-fertilizers+electr...,remind - SSP1-PkBudg650+urease_inhibitors+elec...,2050,potato,climate change,coal tar,1.513394e-06,1.025522e-05
63452,remind - SSP1-PkBudg650+eef-fertilizers+electr...,remind - SSP1-PkBudg650+urease_inhibitors+elec...,2050,potato,climate change,nylon 6-6,1.485724e-06,1.006772e-05
63453,remind - SSP1-PkBudg650+eef-fertilizers+electr...,remind - SSP1-PkBudg650+urease_inhibitors+elec...,2050,potato,climate change,Rest (-),-1.130371e-07,-7.659742e-07


## Add metrics and label

In [6]:
all_results = pd.concat(
    [
        timeline_results[
            ~(
                (timeline_results["year"] == "2050")
                & (timeline_results["scenario"] == background_scenario)
            )
        ],
        full_year_results,
    ]
)
closed_results = close_results(all_results)
closed_results

,scenario_group,scenario,year,impact_category,final_product_name,reference_product_name,value,share
0,baseline,baseline,2025,acidification,cheese,C4 hydrocarbon mixture,0.000002,0.000018
1,baseline,baseline,2025,acidification,cheese,NMC811 hydroxide,0.000000,0.000000
2,baseline,baseline,2025,acidification,cheese,NMC955 hydroxide,0.000000,0.000000
3,baseline,baseline,2025,acidification,cheese,"NOx retained, by selective catalytic reduction",0.000010,0.000075
4,baseline,baseline,2025,acidification,cheese,NPK (15-15-15) fertiliser,0.000090,0.000683
...,...,...,...,...,...,...,...,...
149053,remind - SSP1-PkBudg650+ruminant-methane-mitig...,remind - SSP1-PkBudg650+nop_supplementation,2050,climate change,potato,"wood chips, dry, measured as dry mass",0.000000,0.000000
149054,remind - SSP1-PkBudg650+ruminant-methane-mitig...,remind - SSP1-PkBudg650+nop_supplementation,2050,climate change,potato,"xylene, mixed",0.000000,0.000000
149055,remind - SSP1-PkBudg650+ruminant-methane-mitig...,remind - SSP1-PkBudg650+nop_supplementation,2050,climate change,potato,"zeolite, powder",0.000000,0.000000
149056,remind - SSP1-PkBudg650+ruminant-methane-mitig...,remind - SSP1-PkBudg650+nop_supplementation,2050,climate change,potato,zinc,0.000000,0.000000


In [7]:
for reference_scenario in ["baseline", background_scenario]:
    closed_results = add_activity_metrics_vs_scenario(
        closed_results, reference_scenario=reference_scenario
    )

tractor_activities = [
    activity for activity in ei if "market for tractor" in activity["name"]
]
tractor_consuming_activities = get_consuming_activities(tractor_activities)

tractor_consuming_activities_ref_product = [
    act["reference product"] for act in tractor_consuming_activities
]

final_results = add_label(
    closed_results, tractor_consuming_activities_ref_product, config.feed_products
)

final_results.loc[
    final_results["reference_product_name"].isin(milk_input_products), "is_milk_input"
] = True
final_results.loc[
    final_results["reference_product_name"].isin(potato_input_products),
    "is_potato_input",
] = True

cols_order = [
    "scenario_group",
    "scenario",
    "year",
    "final_product_name",
    "impact_category",
    "reference_product_name",
    "value",
    "share",
    "variation_activity_vs_baseline",
    "variation_activity_vs_baseline_pct",
    "contribution_activity_vs_baseline_pct",
    "variation_activity_vs_background_2050",
    "variation_activity_vs_background_2050_pct",
    "contribution_activity_vs_background_2050_pct",
    "label",
    "is_milk_input",
    "is_potato_input",
]
final_results = final_results[cols_order]

final_results

,scenario_group,scenario,year,final_product_name,impact_category,reference_product_name,value,share,variation_activity_vs_baseline,variation_activity_vs_baseline_pct,contribution_activity_vs_baseline_pct,variation_activity_vs_background_2050,variation_activity_vs_background_2050_pct,contribution_activity_vs_background_2050_pct,label,is_milk_input,is_potato_input
0,baseline,baseline,2025,cheese,acidification,C4 hydrocarbon mixture,0.000002,0.000018,0.000000,0.0,0.000000,-1.524762e-08,-0.006369,-1.291569e-07,C4 hydrocarbon mixture,NaN,NaN
1,baseline,baseline,2025,cheese,acidification,NMC811 hydroxide,0.000000,0.000000,0.000000,0.0,0.000000,-2.220897e-06,-1.000000,-1.881239e-05,NMC811 hydroxide,NaN,NaN
2,baseline,baseline,2025,cheese,acidification,NMC955 hydroxide,0.000000,0.000000,0.000000,0.0,0.000000,-3.279892e-06,-1.000000,-2.778275e-05,NMC955 hydroxide,NaN,NaN
3,baseline,baseline,2025,cheese,acidification,"NOx retained, by selective catalytic reduction",0.000010,0.000075,0.000000,0.0,0.000000,9.814616e-06,0.000000,8.313596e-05,"NOx retained, by selective catalytic reduction",NaN,NaN
4,baseline,baseline,2025,cheese,acidification,NPK (15-15-15) fertiliser,0.000090,0.000683,0.000000,0.0,0.000000,-1.781159e-06,-0.019484,-1.508753e-05,NPK (15-15-15) fertiliser,NaN,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149053,remind - SSP1-PkBudg650+ruminant-methane-mitig...,remind - SSP1-PkBudg650+nop_supplementation,2050,potato,climate change,"wood chips, dry, measured as dry mass",0.000000,0.000000,0.000000,0.0,0.000000,-1.553228e-05,-1.000000,-8.992927e-05,"wood chips, dry, measured as dry mass",True,NaN
149054,remind - SSP1-PkBudg650+ruminant-methane-mitig...,remind - SSP1-PkBudg650+nop_supplementation,2050,potato,climate change,"xylene, mixed",0.000000,0.000000,-0.000004,-1.0,-0.000011,-3.593309e-06,-1.000000,-2.080464e-05,"xylene, mixed",NaN,NaN
149055,remind - SSP1-PkBudg650+ruminant-methane-mitig...,remind - SSP1-PkBudg650+nop_supplementation,2050,potato,climate change,"zeolite, powder",0.000000,0.000000,-0.000006,-1.0,-0.000017,-1.240985e-05,-1.000000,-7.185093e-05,"zeolite, powder",NaN,NaN
149056,remind - SSP1-PkBudg650+ruminant-methane-mitig...,remind - SSP1-PkBudg650+nop_supplementation,2050,potato,climate change,zinc,0.000000,0.000000,-0.000010,-1.0,-0.000032,-9.652749e-06,-1.000000,-5.588777e-05,zinc,NaN,NaN


In [8]:
final_results.sort_values(
    by=["final_product_name", "scenario", "impact_category", "year", "share"],
    ascending=[True, True, True, True, False],
).to_csv(
    f"../Results/{impact_assessment_method}/Formatted/scenario_results.csv", index=False
)

In [ ]:
raw_data_baseline = pd.read_csv(
    "../Results/EF v3.1/Analysis background improvements/EF v3.1_baseline_2025_climate change_no-group.csv"
)

raw_data_2050 = pd.read_csv(
    "../Results/EF v3.1/Analysis background improvements/EF v3.1_remind - SSP1-PkBudg650_2050_climate change_no-group.csv"
)

df_baseline = process_non_grouped_results(raw_data_baseline, "baseline")
df_2050 = process_non_grouped_results(raw_data_2050, "2050")

comparison_df = df_baseline.merge(
    df_2050,
    how="outer",
    on=["final_product_name", "reference product", "name", "location"],
)

comparison_df["variation"] = np.where(
    comparison_df["value_baseline"] != 0,
    (comparison_df["value_2050"] - comparison_df["value_baseline"])
    / comparison_df["value_baseline"],
    "new",
)

comparison_df.rename(
    columns={"reference product": "reference_product_name"}, inplace=True
)

tractor_activities = [
    activity for activity in ei if "market for tractor" in activity["name"]
]
tractor_consuming_activities = get_consuming_activities(tractor_activities)

tractor_consuming_activities_ref_product = [
    act["reference product"] for act in tractor_consuming_activities
]

final_results = add_label(
    comparison_df, tractor_consuming_activities_ref_product, config.feed_products
)

final_results

,final_product_name,reference_product_name,name,location,value_baseline,value_2050,variation,label
0,cheese,"1,1-difluoroethane","1,1-difluoroethane production",RoW,NaN,0.000152,nan,"1,1-difluoroethane"
1,cheese,C4 hydrocarbon mixture,"unsaturated hydrocarbons production, steam cra...",BR,0.000156,0.000157,0.003251089432267906,C4 hydrocarbon mixture
2,cheese,C4 hydrocarbon mixture,"unsaturated hydrocarbons production, steam cra...",CN,0.001333,NaN,nan,C4 hydrocarbon mixture
3,cheese,C4 hydrocarbon mixture,"unsaturated hydrocarbons production, steam cra...",IN,NaN,0.000100,nan,C4 hydrocarbon mixture
4,cheese,C4 hydrocarbon mixture,"unsaturated hydrocarbons production, steam cra...",JP,0.000264,0.000265,0.00453595157725824,C4 hydrocarbon mixture
...,...,...,...,...,...,...,...,...
3030,potato,"zeolite, powder","zeolite production, powder",RoW,0.000004,0.000009,1.2294643695337086,"zeolite, powder"
3031,potato,zinc,primary zinc production from concentrate,CN,NaN,0.000005,nan,zinc
3032,potato,zinc,primary zinc production from concentrate,RoW,0.000010,NaN,nan,zinc
3033,potato,zinc concentrate,zinc mine operation,CN,NaN,0.000003,nan,zinc concentrate


In [7]:
final_results.sort_values(
    by=["final_product_name", "reference_product_name", "location"]
).to_csv(
    f"../Results/{impact_assessment_method}/Formatted/analysis_background_improvements.csv",
    index=False,
)